# 03 — Build Dataset Arrays

**Input datasets:** `lemon-meta` + `lemon-features`

Stacks per-subject `.npy` files into training-ready arrays and computes targets.

**Outputs (save as `lemon-dataset`):**
- `X.npy` — `(N, 2, 48, 48, 32)` brain volumes
- `y_regression.npy` — `(N, 3)` z-scored wellbeing targets
- `y_network.npy` — `(N, 3)` DMN/Salience/ECN activation 0–1
- `wb_stats.json` — mean/std for inverting z-scores at inference
- `net_scaler.pkl` — MinMaxScaler for network targets

In [ ]:
!pip install -q scikit-learn

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd

META_DIR = '/kaggle/input/lemon-meta'
FEAT_DIR = '/kaggle/input/lemon-features/features'

pheno    = pd.read_csv(f'{META_DIR}/phenotypic.csv')
with open(f'{META_DIR}/target_columns.json') as f:
    col_map = json.load(f)

feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('.npy')])
print(f'Feature files: {len(feat_files)}')
print(f'Target columns: {col_map}')

In [ ]:
# ── ROI seed masks for network activation labels ───────────────────────────────
TARGET_AFFINE = np.diag([4.0, 4.0, 4.5, 1.0])
TARGET_SHAPE  = (48, 48, 32)

NETWORK_SEEDS = {
    'DMN':      [(-1, 52, -6),   (-1, -53, 26)],
    'Salience': [(38, 6, -4),    (4, 28, 26)],
    'ECN':      [(-44, 28, 28),  (-30, -56, 46)],
}

def mni_to_vox(coord, affine):
    vox = (np.linalg.inv(affine) @ np.array([*coord, 1]))[:3]
    return np.clip(np.round(vox).astype(int), 0, np.array(TARGET_SHAPE)-1)

def sphere_mask(center, r, shape):
    x, y, z = np.mgrid[:shape[0], :shape[1], :shape[2]]
    return np.sqrt((x-center[0])**2+(y-center[1])**2+(z-center[2])**2) <= r

seed_masks = {}
for net, seeds in NETWORK_SEEDS.items():
    masks = [sphere_mask(mni_to_vox(c, TARGET_AFFINE), 1.5, TARGET_SHAPE) for c in seeds]
    seed_masks[net] = np.any(masks, axis=0)
    print(f'{net}: {seed_masks[net].sum()} voxels')

print('Seed masks ready ✓')

In [ ]:
from tqdm import tqdm

pheno_idx = pheno.set_index('participant_id')
X_list, y_reg_list, y_net_list, subject_ids, skipped = [], [], [], [], []

for fname in tqdm(feat_files):
    sub = fname.replace('.npy', '')
    pid = sub if sub.startswith('sub-') else f'sub-{sub}'

    if pid not in pheno_idx.index:
        skipped.append(sub); continue

    row     = pheno_idx.loc[pid]
    targets = [row.get(col_map.get(k), np.nan)
               for k in ['trait_anxiety', 'chronic_stress', 'neuroticism']]
    if any(np.isnan(t) for t in targets):
        skipped.append(sub); continue

    feats = np.load(f'{FEAT_DIR}/{fname}')     # (2, 48, 48, 32)
    falff = feats[1]
    net_vals = [float(falff[seed_masks[n]].mean()) for n in ['DMN', 'Salience', 'ECN']]

    X_list.append(feats)
    y_reg_list.append(targets)
    y_net_list.append(net_vals)
    subject_ids.append(sub)

print(f'Included: {len(X_list)}   Skipped: {len(skipped)}')

In [ ]:
from sklearn.preprocessing import MinMaxScaler

X         = np.array(X_list,     dtype=np.float32)
y_reg_raw = np.array(y_reg_list, dtype=np.float32)
y_net_raw = np.array(y_net_list, dtype=np.float32)

# Z-score regression targets
wb_mean = y_reg_raw.mean(axis=0)
wb_std  = y_reg_raw.std(axis=0)
y_reg   = ((y_reg_raw - wb_mean) / (wb_std + 1e-8)).astype(np.float32)

wb_stats = {
    'targets': ['trait_anxiety', 'chronic_stress', 'neuroticism'],
    'mean':    wb_mean.tolist(),
    'std':     wb_std.tolist(),
}
with open('/kaggle/working/wb_stats.json', 'w') as f:
    json.dump(wb_stats, f, indent=2)

# Scale network targets 0–1
scaler = MinMaxScaler()
y_net  = scaler.fit_transform(y_net_raw).astype(np.float32)
with open('/kaggle/working/net_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save arrays
np.save('/kaggle/working/X.npy',            X)
np.save('/kaggle/working/y_regression.npy', y_reg)
np.save('/kaggle/working/y_network.npy',    y_net)
with open('/kaggle/working/subject_ids_final.json', 'w') as f:
    json.dump(subject_ids, f)

print(f'X:     {X.shape}')
print(f'y_reg: {y_reg.shape}')
print(f'y_net: {y_net.shape}')
print(f'Dataset size: {(X.nbytes+y_reg.nbytes+y_net.nbytes)/1e6:.0f} MB')
print()
print('NEXT STEPS:')
print('  Save Version → "Save as Dataset" → name it "lemon-dataset"')
print('  Then open notebook 04 and attach lemon-dataset')